In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [3]:
!pip install dagshub
!pip install mlflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 2.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 5.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: dacite
    Found existing installation: dacite 1.9.2
    Uninstalling dacite-1.9.2:
      Successfully uninstalled dacite-1.9.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires dacite<2,>=1.9, but you have dacite 1.6.0 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 689.8 kB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.3 MB/s eta 0:00

In [4]:
import sys
sys.path.append('/kaggle/usr/lib/notebooks/nikolozdodashvili/preprocessing/notebooks/nikolozdodashvili')

from preprocessing import (
    optimize_memory,
    DropHighMissingFeatures,
    FrequencyEncoder,
    MissingValueImputer,
    TimeFeatureExtractor,
    TransactionAmtTransformer,
    GroupAggregator,
    DropCorrelatedFeatures
)

In [5]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity    = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how='left')

X = train.drop(columns=['isFraud', 'TransactionID'])
y = train['isFraud'].astype('int8')

X = optimize_memory(X)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

მეხსიერება დამუშავებამდე: 1946.36 MB
მეხსიერება დამუშავების შემდეგ: 921.47 MB
შემცირდა: 52.7%


In [6]:
from sklearn.base import BaseEstimator, TransformerMixin

class FrequencyEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.frequency_maps = {}

    def fit(self, X, y=None):
        cat_cols = X.select_dtypes(include=['object', 'category']).columns
        for col in cat_cols:
            self.frequency_maps[col] = X[col].value_counts(normalize=True).to_dict()
        return self

    def transform(self, X):
        X = X.copy()
        for col, freq_map in self.frequency_maps.items():
            if col in X.columns:
                if hasattr(X[col], 'cat'):
                    X[col] = X[col].astype('object')
                X[col] = X[col].map(freq_map).fillna(0)
        return X

# TRAIN AND MLFLOW

In [7]:
import mlflow
import mlflow.sklearn
import dagshub
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score

dagshub.init(repo_owner='ndoda23',
             repo_name='MachineLearning---IEEE-CIS-Fraud-Detection',
             mlflow=True)

mlflow.set_experiment("LightGBM_Training")

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=098df2fe-1ebd-4aef-af31-7ce58f21666f&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=2d9976ec45abe2cfa012dbe2b5e33e09770d67f317904873d096a66ba004934c




Accessing as ndoda23

Initialized MLflow to track repo "ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection"

Repository ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection initialized!

<Experiment: artifact_location='mlflow-artifacts:/828ea78bbc4e4d58b2e34f214d5fb6ff', creation_time=1778068558591, experiment_id='2', last_update_time=1778068558591, lifecycle_stage='active', name='LightGBM_Training', tags={}, trace_location=None, workspace='default'>

In [8]:
# ===== Cleaning Run =====
with mlflow.start_run(run_name="LightGBM_Cleaning"):
    cleaner = DropHighMissingFeatures(threshold=0.9)
    cleaner.fit(X_train)
    mlflow.log_param("missing_threshold", 0.9)
    mlflow.log_metric("dropped_columns", len(cleaner.features_to_drop_))
    mlflow.log_metric("remaining_columns", X_train.shape[1] - len(cleaner.features_to_drop_))

Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
🏃 View run LightGBM_Cleaning at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/5af122a6557f4cad8c500ed8b2d3c949
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [9]:
# ===== Feature Engineering Run =====
with mlflow.start_run(run_name="LightGBM_Feature_Engineering"):
    mlflow.log_param("encoding", "FrequencyEncoding")
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("time_features", True)
    mlflow.log_param("amt_features", True)
    mlflow.log_param("group_aggregations", "card1, addr1")
    mlflow.log_metric("features_added", 13)

🏃 View run LightGBM_Feature_Engineering at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/5fc889ae4cf4404c98d8665724015f6b
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [12]:
with mlflow.start_run(run_name="LightGBM_Training_run1"):
    
    pipeline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('time',     TimeFeatureExtractor()),
        ('amt',      TransactionAmtTransformer()),
        ('group',    GroupAggregator()),
        ('selector', DropCorrelatedFeatures(threshold=0.92)),
        ('model',    LGBMClassifier(
            n_estimators=300, 
            learning_rate=0.05,
            max_depth=6, 
            random_state=42,
            verbose=-1,
            importance_type='gain'
        ))
    ])

    print("Training pipeline...")
    pipeline.fit(X_train, y_train)

    y_train_proba = pipeline.predict_proba(X_train)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_proba)

    y_val_proba = pipeline.predict_proba(X_val)[:, 1]
    y_val_class = pipeline.predict(X_val)
    val_auc = roc_auc_score(y_val, y_val_proba)

    print("Running Cross-Validation...")
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)

    metrics = {
        "train_roc_auc": train_auc,     
        "val_roc_auc": val_auc,
        "train_val_diff": train_auc - val_auc, 
        "val_pr_auc": average_precision_score(y_val, y_val_proba),
        "val_f1": f1_score(y_val, y_val_class),
        "cv_auc_mean": cv_scores.mean(),
        "cv_auc_std": cv_scores.std()
    }
    mlflow.log_metrics(metrics)

    mlflow.log_params({"n_estimators": 300, "learning_rate": 0.05, "max_depth": 6})
    mlflow.sklearn.log_model(pipeline, "fraud_detection_pipeline")

    print("-" * 30)
    print(f"✅ Run Completed!")
    print(f"Train ROC AUC: {train_auc:.4f}")
    print(f"Validation ROC AUC: {val_auc:.4f}")
    print(f"Difference (Overfit): {train_auc - val_auc:.4f}")
    print(f"CV ROC AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Training pipeline...
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 165
დარჩება 268 სვეტი
Running Cross-Validation...


2026/05/06 15:56:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 15:56:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


------------------------------
✅ Run Completed!
Train ROC AUC: 0.9401
Validation ROC AUC: 0.9290
Difference (Overfit): 0.0111
CV ROC AUC: 0.9265 ± 0.0016
🏃 View run LightGBM_Training_run1 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/cc5ab3193bc447c0b3a751a8808b9eb5
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2


In [13]:
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, precision_score, recall_score, classification_report
from sklearn.model_selection import cross_val_score
import mlflow.sklearn

# Run 2 - minus999 სტრატეგია და მეტი estimators
with mlflow.start_run(run_name="LightGBM_Training_run2"):
    
    pipeline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='minus999')), # სტრატეგიის ცვლილება
        ('time',     TimeFeatureExtractor()),
        ('amt',      TransactionAmtTransformer()),
        ('group',    GroupAggregator()),
        ('selector', DropCorrelatedFeatures(threshold=0.92)),
        ('model',    LGBMClassifier(
            n_estimators=500,  # გაზრდილი რაოდენობა
            learning_rate=0.05,
            max_depth=6, 
            random_state=42,
            verbose=-1,
            importance_type='gain'
        ))
    ])

    print("Training pipeline for Run 2...")
    pipeline.fit(X_train, y_train)

    # პროგნოზები წვრთნის მონაცემებზე (Train metrics)
    y_train_proba = pipeline.predict_proba(X_train)[:, 1]
    train_auc = roc_auc_score(y_train, y_train_proba)

    # პროგნოზები ვალიდაციის მონაცემებზე (Validation metrics)
    y_val_proba = pipeline.predict_proba(X_val)[:, 1]
    y_val_class = pipeline.predict(X_val)
    val_auc = roc_auc_score(y_val, y_val_proba)

    # Cross-validation
    print("Running Cross-Validation (3-fold)...")
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc', n_jobs=-1)

    # MLflow Params
    mlflow.log_params({
        "n_estimators": 500,
        "learning_rate": 0.05,
        "max_depth": 6,
        "imputer_strategy": "minus999",
        "corr_threshold": 0.92
    })

    # MLflow Metrics (Train + Val + CV)
    mlflow.log_metrics({
        "train_roc_auc": train_auc,  # წვრთნის ქულა
        "val_roc_auc": val_auc,
        "val_pr_auc": average_precision_score(y_val, y_val_proba),
        "val_f1": f1_score(y_val, y_val_class),
        "cv_auc_mean": cv_scores.mean(),
        "cv_auc_std": cv_scores.std()
    })

    mlflow.sklearn.log_model(pipeline, "pipeline_run2")

    print("-" * 30)
    print(f"✅ Run 2 Completed!")
    print(f"Train ROC AUC: {train_auc:.4f}")
    print(f"Validation ROC AUC: {val_auc:.4f}")
    print(f"CV ROC AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

Training pipeline for Run 2...
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 167
დარჩება 266 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 170
დარჩება 263 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 163
დარჩება 270 სვეტი
Threshold 0.92: წაიშლება 338
დარჩება 95 სვეტი
Running Cross-Validation (3-fold)...


2026/05/06 16:06:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 16:06:53 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


------------------------------
✅ Run 2 Completed!
Train ROC AUC: 0.9564
Validation ROC AUC: 0.9414
CV ROC AUC: 0.9365 ± 0.0008
🏃 View run LightGBM_Training_run2 at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2/runs/81b0743696c54b579570db25a642dfda
🧪 View experiment at: https://dagshub.com/ndoda23/MachineLearning---IEEE-CIS-Fraud-Detection.mlflow/#/experiments/2
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 340
დარჩება 93 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 338
დარჩება 95 სვეტი
Threshold 90%: 12 სვეტი წაიშლება
დარჩენილი სვეტები: 420
Threshold 0.92: წაიშლება 337
დარჩება 96 სვეტი


In [ ]:
# Run 3 - num_leaves მეტი, deeper
with mlflow.start_run(run_name="LightGBM_Training_run3"):
    pipeline = Pipeline([
        ('cleaner',  DropHighMissingFeatures(threshold=0.9)),
        ('encoder',  FrequencyEncoder()),
        ('imputer',  MissingValueImputer(strategy='median')),
        ('time',     TimeFeatureExtractor()),
        ('amt',      TransactionAmtTransformer()),
        ('group',    GroupAggregator()),
        ('selector', DropCorrelatedFeatures(threshold=0.92)),
        ('model',    LGBMClassifier(n_estimators=300, learning_rate=0.01,
                                    max_depth=8, num_leaves=63,
                                    random_state=42, verbose=-1))
    ])
    pipeline.fit(X_train, y_train)

    y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
    y_pred_class = pipeline.predict(X_val)
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=3, scoring='roc_auc')

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.01)
    mlflow.log_param("max_depth", 8)
    mlflow.log_param("num_leaves", 63)
    mlflow.log_param("imputer_strategy", "median")
    mlflow.log_param("corr_threshold", 0.92)
    mlflow.log_metric("val_roc_auc",   roc_auc_score(y_val, y_pred_proba))
    mlflow.log_metric("val_pr_auc",    average_precision_score(y_val, y_pred_proba))
    mlflow.log_metric("val_f1",        f1_score(y_val, y_pred_class))
    mlflow.log_metric("val_precision", precision_score(y_val, y_pred_class))
    mlflow.log_metric("val_recall",    recall_score(y_val, y_pred_class))
    mlflow.log_metric("cv_auc_mean",   cv_scores.mean())
    mlflow.log_metric("cv_auc_std",    cv_scores.std())
    mlflow.sklearn.log_model(pipeline, "pipeline")

    print(f"Run 3: ROC AUC={roc_auc_score(y_val, y_pred_proba):.4f}, CV={cv_scores.mean():.4f} ± {cv_scores.std():.4f}")